In [29]:
# 01_sales_forecasting/eda_diagnostics.py
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

config_file = Path("..") / "config.yaml"
with open(config_file, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_paths = config["paths"]
raw_dir = Path(data_paths["base_dir"])
train_path = raw_dir / data_paths["train"]
test_path = raw_dir / data_paths["test"]
oil_path = raw_dir / data_paths["oil"]
stores_path = raw_dir / data_paths["stores"]
holidays_path = raw_dir / data_paths["holidays"]
transactions_path = raw_dir / data_paths["transactions"]
output_submission_path = Path(data_paths["output_submission"])

In [30]:
print("--- Loading Datasets ---")
# Load metadata tables fully
oil = pd.read_csv(oil_path)
stores = pd.read_csv(stores_path)
holidays = pd.read_csv(holidays_path)
transactions = pd.read_csv(transactions_path)
test = pd.read_csv(test_path)

# Load just a chunk of train to inspect schemas safely without memory strain
train_chunk = pd.read_csv(train_path, nrows=1000)

--- Loading Datasets ---


In [31]:
print("\n[1] Columns and Data Types:")
print("Train columns:", list(train_chunk.columns))
print("Test columns:", list(test.columns))
print("Oil columns:", list(oil.columns))
print("Holidays columns:", list(holidays.columns))


[1] Columns and Data Types:
Train columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
Test columns: ['id', 'date', 'store_nbr', 'family', 'onpromotion']
Oil columns: ['date', 'dcoilwtico']
Holidays columns: ['date', 'type', 'locale', 'locale_name', 'description', 'transferred']


In [32]:
print("\n[2] Missing Value Summary (Null Counts):")
print(f"Oil Nulls:\n{oil.isnull().sum()}")
print(f"Holidays Nulls:\n{holidays.isnull().sum()}")
print(f"Transactions Nulls:\n{transactions.isnull().sum()}")


[2] Missing Value Summary (Null Counts):
Oil Nulls:
date           0
dcoilwtico    43
dtype: int64
Holidays Nulls:
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64
Transactions Nulls:
date            0
store_nbr       0
transactions    0
dtype: int64


In [33]:
print("\n[3] Temporal Coverage (Date Bounds):")
# Parse dates for metadata
oil['date'] = pd.to_datetime(oil['date'])
test['date'] = pd.to_datetime(test['date'])
transactions['date'] = pd.to_datetime(transactions['date'])

print(f"Test Date Range:         {test['date'].min().strftime('%Y-%m-%d')} to {test['date'].max().strftime('%Y-%m-%d')}")
print(f"Oil Date Range:          {oil['date'].min().strftime('%Y-%m-%d')} to {oil['date'].max().strftime('%Y-%m-%d')}")
print(f"Transactions Date Range: {transactions['date'].min().strftime('%Y-%m-%d')} to {transactions['date'].max().strftime('%Y-%m-%d')}")


[3] Temporal Coverage (Date Bounds):
Test Date Range:         2017-08-16 to 2017-08-31
Oil Date Range:          2013-01-01 to 2017-08-31
Transactions Date Range: 2013-01-01 to 2017-08-15


In [34]:
# For train.csv, read the first row and the last row to get exact date bounds efficiently
first_row = pd.read_csv(train_path, nrows=1)
# Estimate file size to read tail efficiently
row_count = sum(1 for _ in open(train_path)) - 1
last_row = pd.read_csv(train_path, skiprows=row_count, header=None, names=first_row.columns)
print(f"Train Date Range:        {first_row['date'].values[0]} to {last_row['date'].values[0]}")
print(f"Total Rows in Train:     {row_count:,}")

Train Date Range:        2013-01-01 to 2017-08-15
Total Rows in Train:     3,000,888


In [35]:
# 1. Inspect Oil Data Gaps
oil['date'] = pd.to_datetime(oil['date'])
total_days = (oil['date'].max() - oil['date'].min()).days + 1
missing_dates = total_days - oil['date'].nunique()
null_prices = oil['dcoilwtico'].isnull().sum()

print("[Oil Telemetry Anomalies]")
print(f"  - Unique calendar days recorded: {oil['date'].nunique()} out of {total_days} total span days.")
print(f"  - Calendar dates entirely missing from file (weekends/holidays): {missing_dates}")
print(f"  - Rows present in file but missing a price (NaN): {null_prices}")
print(f"  - Total implicit + explicit gaps to interpolate: {missing_dates + null_prices}\n")

[Oil Telemetry Anomalies]
  - Unique calendar days recorded: 1218 out of 1704 total span days.
  - Calendar dates entirely missing from file (weekends/holidays): 486
  - Rows present in file but missing a price (NaN): 43
  - Total implicit + explicit gaps to interpolate: 529



In [36]:
# 2. Inspect Holiday Realities
print("[Holiday Metadata Pitfalls]")
print(f"  - Total unique holiday/event listings: {len(holidays)}")
print(f"  - Count of officially 'transferred' holidays: {holidays['transferred'].sum()}")
print("  - Break down of event types:")
print(holidays['type'].value_counts())
print("")

[Holiday Metadata Pitfalls]
  - Total unique holiday/event listings: 350
  - Count of officially 'transferred' holidays: 12
  - Break down of event types:
type
Holiday       221
Event          56
Additional     51
Transfer       12
Bridge          5
Work Day        5
Name: count, dtype: int64



In [37]:
# 3. Test Horizon Mechanics
test['date'] = pd.to_datetime(test['date'])
test_horizon_days = (test['date'].max() - test['date'].min()).days + 1
print("[Inference Horizon Scope]")
print(f"  - Blind test window spans from: {test['date'].min().strftime('%Y-%m-%d')} to {test['date'].max().strftime('%Y-%m-%d')}")
print(f"  - Exact number of prediction forecast days (Horizon): {test_horizon_days} days")
print(f"  - Unique stores to forecast: {test['store_nbr'].nunique()}")
print(f"  - Unique product families to forecast: {test['family'].nunique()}")
print(f"  - Total unique time-series combinations: {test['store_nbr'].nunique() * test['family'].nunique()}\n")

[Inference Horizon Scope]
  - Blind test window spans from: 2017-08-16 to 2017-08-31
  - Exact number of prediction forecast days (Horizon): 16 days
  - Unique stores to forecast: 54
  - Unique product families to forecast: 33
  - Total unique time-series combinations: 1782



In [38]:
# 01_sales_forecasting/eda_visualizations.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [39]:
# Set visual styling for clean presentation
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})

In [40]:
print(" Initializing Data Visualization Pipeline...")

# Ensure save directory exists
os.makedirs("eda_plots", exist_ok=True)

oil['date'] = pd.to_datetime(oil['date'])

# Load a substantial chunk of train to evaluate distribution securely (e.g., 1.5 million rows)
print(" Streaming training sample for distribution profiling...")
train_sample = pd.read_csv(train_path, nrows=1500000)

# -------------------------------------------------------------
# PLOT 1: Target Transformation (Handling Long-Tails & Volatility)
# -------------------------------------------------------------
print(" Generating Plot 1: Target Distributions...")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw Sales Distribution
sns.histplot(train_sample['sales'], bins=50, ax=axes[0], color='crimson', kde=False)
axes[0].set_title("Raw Sales Target Distribution\n(Highly Skewed / Zero-Inflated)")
axes[0].set_xlabel("Sales Units")
axes[0].set_ylabel("Frequency Count")
axes[0].set_yscale('log') # Log-scaled axis to see the extreme tail

# Log1p Transformed Sales Distribution
log_sales = np.log1p(train_sample['sales'])
sns.histplot(log_sales, bins=50, ax=axes[1], color='teal', kde=False)
axes[1].set_title("Log-Transformed Target Space: ln(Sales + 1)\n(Normalizes Scales for RMSE optimization)")
axes[1].set_xlabel("ln(Sales + 1)")
axes[1].set_ylabel("Frequency Count")

plt.tight_layout()
plt.savefig("eda_plots/01_target_transformation.png", dpi=150)
plt.close()

# -------------------------------------------------------------
# PLOT 2: Imputing Macroeconomic Telemetry (Oil Market Gaps)
# -------------------------------------------------------------
print(" Generating Plot 2: Oil Price Imputation Strategy...")
# Isolate a snapshot window (e.g., First half of 2013) to see the gaps up close
oil_snapshot = oil[(oil['date'] >= '2013-01-01') & (oil['date'] <= '2013-04-01')].copy()

# Create continuous daily baseline tracking framework
full_range = pd.date_range(start=oil_snapshot['date'].min(), end=oil_snapshot['date'].max(), freq='D')
oil_filled = oil_snapshot.set_index('date').reindex(full_range).ffill().bfill().reset_index()
oil_filled.columns = ['date', 'oil_price']

plt.figure(figsize=(12, 5))
plt.plot(oil_filled['date'], oil_filled['oil_price'], label='Forward-Filled Continuous Trend', color='navy', linestyle='-', alpha=0.8)
plt.scatter(oil_snapshot['date'], oil_snapshot['dcoilwtico'], label='Raw Available Market Days', color='orange', s=25, zorder=3)

plt.title("Oil Price Interpolation Strategy (Macroeconomic Baseline Gaps)")
plt.xlabel("Timeline Index")
plt.ylabel("WTI Crude Oil Price ($/Barrel)")
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig("eda_plots/02_oil_imputation.png", dpi=150)
plt.close()

# -------------------------------------------------------------
# PLOT 3: Product Family Zero-Inflation Matrix
# -------------------------------------------------------------
print(" Generating Plot 3: Zero-Sales Inflation by Product Class...")
# Calculate zero percentage per family
family_stats = train_sample.groupby('family')['sales'].agg(
    total_rows='count',
    zero_rows=lambda x: (x == 0).sum()
).reset_index()
family_stats['zero_percentage'] = (family_stats['zero_rows'] / family_stats['total_rows']) * 100
family_stats = family_stats.sort_values(by='zero_percentage', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=family_stats, x='zero_percentage', y='family', palette='viridis', hue='family', legend=False)
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50% Zero Threshold Boundary')

plt.title("Proportion of Absolute Zero-Sales Entries by Product Family")
plt.xlabel("Percentage of Records with 0.0 Sales (%)")
plt.ylabel("Product Family Classification")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("eda_plots/03_zero_inflation_by_family.png", dpi=150)
plt.close()

print("\n Visualizations compiled perfectly! Images saved inside the 'eda_plots/' folder:")
print("  eda_plots/01_target_transformation.png")
print("  eda_plots/02_oil_imputation.png")
print("  eda_plots/03_zero_inflation_by_family.png")

 Initializing Data Visualization Pipeline...
 Streaming training sample for distribution profiling...
 Generating Plot 1: Target Distributions...
 Generating Plot 2: Oil Price Imputation Strategy...
 Generating Plot 3: Zero-Sales Inflation by Product Class...

 Visualizations compiled perfectly! Images saved inside the 'eda_plots/' folder:
  eda_plots/01_target_transformation.png
  eda_plots/02_oil_imputation.png
  eda_plots/03_zero_inflation_by_family.png


In [41]:
# 01_sales_forecasting/data_split.py
# Read core files
train = pd.read_csv(train_path)
train['date'] = pd.to_datetime(train['date'])

# 1. Define your strict validation cutoff based on our 16-day horizon calculation
val_cutoff = pd.to_datetime("2017-08-01")

# 2. Partition datasets chronologically
train_split = train[train['date'] < val_cutoff]
val_split = train[train['date'] >= val_cutoff]

# 3. Apply the log transformation to check scale transformations
y_train_log = np.log1p(train_split['sales'])
y_val_log = np.log1p(val_split['sales'])

print("\n [Data Split Verification Results]")
print(f"  - Full Train Records:      {len(train):,}")
print(f"  - Isolated Train Split:    {len(train_split):,} rows (Min Date: {train_split['date'].min().strftime('%Y-%m-%d')}, Max Date: {train_split['date'].max().strftime('%Y-%m-%d')})")
print(f"  - Isolated Validation Split: {len(val_split):,} rows (Min Date: {val_split['date'].min().strftime('%Y-%m-%d')}, Max Date: {val_split['date'].max().strftime('%Y-%m-%d')})")
print(f"  - Log Transformed Target Mean (Train): {y_train_log.mean():.4f}")
print(f"  - Log Transformed Target Mean (Val):   {y_val_log.mean():.4f}")




 [Data Split Verification Results]
  - Full Train Records:      3,000,888
  - Isolated Train Split:    2,974,158 rows (Min Date: 2013-01-01, Max Date: 2017-07-31)
  - Isolated Validation Split: 26,730 rows (Min Date: 2017-08-01, Max Date: 2017-08-15)
  - Log Transformed Target Mean (Train): 2.9201
  - Log Transformed Target Mean (Val):   3.6291
